In [1]:
import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from matplotlib.ticker import FuncFormatter
from matplotlib.dates import *

In [2]:
from config import DRIVER as driver

In [3]:
def to_list(cursor):
    return list(map(dict, cursor))

def to_data_frame(cursor):
    return pd.DataFrame(to_list(cursor))

def get_transactions(ind, am):
    url = "http://127.0.0.1:28081/get_outs"
    # Define the JSON payload
    payload = {"outputs":[{"amount": am, "index":ind}], "get_txid": True}
    # Define the headers
    headers = {
        "Content-Type": "application/json"
    }

    # Make the POST request
    response = requests.post(url, json=payload, headers=headers)
    return response.json()

First we will get the number of inputs with a mix in equal to 0. Remember this data has been creaed in a local tesnet, so the number with mixin equal to 0 may be elevated to show the privicy risk of using mixin 0. Rememebre that mixin 0 is not recommended for privacy reasons and you can no longer use it in the Monero main network.

In [4]:
with driver.session() as session:
    num_inputs = to_list(session.run(query=  """MATCH (i:Input) RETURN count(i)"""))[0]['count(i)']
    num_inputs_mix_0 = to_list(session.run(query=  """MATCH (i:Input) WHERE i.mixin = '0' RETURN count(i)"""))[0]['count(i)']

print(f"Number of inputs: {num_inputs}")
print(f"\nNumber of inputs with mixin equal to 0: {num_inputs_mix_0}. This inputs do not have the benefits of ring signatures")

Number of inputs: 1721

Number of inputs with mixin equal to 0: 766. This inputs do not have the benefits of ring signatures


Our next step is to get all the information of those inputs with mixins equal to 0 to see what information we can get from them.

In [5]:
query = """MATCH (i:Input)-[:TX_INPUT]->(t:Transactions)
            WHERE i.mixin = '0'
            RETURN i, t"""

with driver.session() as session:
    data = to_list(session.run(query=query))
    
    rows = []
    for record in data:
        input_node = record['i']
        transaction_node = record['t']
        row = {
                    'input_mixin': input_node.get('mixin'),
                    'input_anonset': input_node.get('anonset'),
                    'input_id': input_node.get('id'),
                    'input_value': input_node.get('value'),
                    'key_offset': input_node.get('key_offset'),
                    'transaction_fee': transaction_node.get('fee'),
                    'transaction_id': transaction_node.get('id'),
                    'transaction_hash': transaction_node.get('hash')
                }
        rows.append(row)
inputs = pd.DataFrame(rows)
inputs

,input_mixin,input_anonset,input_id,input_value,key_offset,transaction_fee,transaction_id,transaction_hash
0,0,63,i0,90000000000,[3],4268127321,t65,fbc9c915ebe08091e4bd58d039289c169dd1852a481715...
1,0,64,i1,10000000000000,[2],4268127321,t65,fbc9c915ebe08091e4bd58d039289c169dd1852a481715...
2,0,64,i2,10000000000000,[1],4268127321,t65,fbc9c915ebe08091e4bd58d039289c169dd1852a481715...
3,0,65,i3,7000000000000,[0],2458473645,t66,059b07e3ab70b55b93339026dd021e3bc5f1c65d70b11b...
4,0,65,i4,10000000000000,[3],3212403907,t67,6d6377e0fa4143181e7ac1b36216ad7e3d50a4309a7c43...
...,...,...,...,...,...,...,...,...
761,0,26,i761,9000000000000,[19],4946746849,t390,3115a8abbc0fe5b14b7ba1c996957df72c8703809337ef...
762,0,334,i762,10000000000000,[63],4946746849,t390,3115a8abbc0fe5b14b7ba1c996957df72c8703809337ef...
763,0,216,i763,90000000000,[144],4946746849,t390,3115a8abbc0fe5b14b7ba1c996957df72c8703809337ef...
764,0,59,i764,600000000000,[36],4946746849,t390,3115a8abbc0fe5b14b7ba1c996957df72c8703809337ef...


Notice that there are key_offset that repeat. This is because the key_offset it point us at the output of some previous block, and that block can have multiple output. Is the key (which is an information will get on the next steps) along with the key_offset that will help us to identify the output that is being spent.

In [16]:
k_offsets_mix_0 = {}
tx_used = []
for _, row in inputs[['input_value', 'key_offset', 'transaction_hash']].iterrows():
    key_offset = row.key_offset.split('[')[1].split(']')[0]
    value = row.input_value
    outs = get_transactions(int(key_offset), int(value))
    if key_offset not in k_offsets_mix_0.keys():
        k_offsets_mix_0[key_offset] = [{'block': outs['outs'][0]['height'], 'key': outs['outs'][0]['key'], 'amount': value }]
    else:
        k_offsets_mix_0[key_offset].append({'block': outs['outs'][0]['height'], 'key': outs['outs'][0]['key'], 'amount': value })
    tx_used.append(outs['outs'][0]['key'])

With this last dictionary, we have the information of each key_offset used in the transactions with mixin equal to 0. As the mixins is 0, the transactions that appears is 100% the one being spend. If this key it is used in another transaction, we will know that it is not the real one as it has been previosuly spent.

In [7]:
query = """MATCH (i:Input)-[:TX_INPUT]->(t:Transactions)
            WHERE i.mixin <> '0'
            RETURN i, t"""

with driver.session() as session:
    data = to_list(session.run(query=query))
    
    rows = []
    for record in data:
        input_node = record['i']
        transaction_node = record['t']
        row = {
                    'input_mixin': input_node.get('mixin'),
                    'input_anonset': input_node.get('anonset'),
                    'input_id': input_node.get('id'),
                    'input_value': input_node.get('value'),
                    'key_offset': input_node.get('key_offset'),
                    'transaction_fee': transaction_node.get('fee'),
                    'transaction_id': transaction_node.get('id'),
                    'transaction_hash': transaction_node.get('hash')
                }
        rows.append(row)
inputs_mul_mix = pd.DataFrame(rows)
inputs_mul_mix

,input_mixin,input_anonset,input_id,input_value,key_offset,transaction_fee,transaction_id,transaction_hash
0,1,28,i766,4000000000000,"[21, 3]",2812564057,t407,53d0a3e54acaab8d3083e72d81b9afdeded7081894b1ca...
1,1,351,i767,10000000000000,"[182, 126]",2812564057,t407,53d0a3e54acaab8d3083e72d81b9afdeded7081894b1ca...
2,1,353,i769,10000000000000,"[232, 47]",4995647194,t410,3120ca4227a2845f24882dd55410354b9b01f626f4f0f5...
3,1,38,i770,700000000000,"[20, 5]",4995647194,t410,3120ca4227a2845f24882dd55410354b9b01f626f4f0f5...
4,1,47,i768,7000000000,"[15, 19]",4995647194,t410,3120ca4227a2845f24882dd55410354b9b01f626f4f0f5...
...,...,...,...,...,...,...,...,...
950,9,57,i1715,4000000000000,"[3, 21, 4, 5, 1, 3, 1, 1, 1, 6]",10602633647,t927,f4e922717d99f32a7b4564348af0e7c715ce71b29341cc...
951,9,135,i1719,5000000000,"[23, 5, 4, 39, 4, 20, 5, 6, 17, 9]",10602633647,t927,f4e922717d99f32a7b4564348af0e7c715ce71b29341cc...
952,9,808,i1718,10000000000000,"[203, 97, 18, 15, 90, 19, 14, 23, 85, 48]",10602633647,t927,f4e922717d99f32a7b4564348af0e7c715ce71b29341cc...
953,9,622,i1716,500000000000,"[128, 116, 48, 54, 95, 34, 8, 7, 13, 18]",10602633647,t927,f4e922717d99f32a7b4564348af0e7c715ce71b29341cc...


In [8]:
total_mixins = {'1': 0, '3': 0, '9': 0} 
vul_mixins = {'1': 0, '3': 0, '9': 0}
for _, row in inputs_mul_mix[['input_value', 'key_offset', 'transaction_hash', 'input_mixin']].iterrows():
    key_offset = row.key_offset.split('[')[1].split(']')[0].split(',')
    value = row.input_value
    num_mixin = row.input_mixin
    for i, key in enumerate(key_offset):
        total_mixins[str(num_mixin)] += 1
        key_i = int(key)
        out = get_transactions(key_i, int(value))
        if out['outs'][0]['key'] in tx_used:
            if num_mixin == 1:
                # If a transaction with 2 inputs uses one of the used inputs of our list,
                # we can also addthe other input in the list since it will be another used output
                tx_used.append(get_transactions(int(key_offset[1-i]), int(value))['outs'][0]['key'])
            vul_mixins[str(num_mixin)] += 1
            4
            print('-----------')
            print('Vul found')

-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
----------

In [10]:
print("Persentage of deducible mixins with ring size equal to 2: ", vul_mixins['1']/total_mixins['1'])
print("Persentage of deducible mixins with ring size equal to 4: ", vul_mixins['3']/total_mixins['3'])
print("Persentage of deducible mixins with ring size equal to 10: ", vul_mixins['9']/total_mixins['9'])

Persentage of deducible mixins with ring size equal to 2:  0.3106060606060606
Persentage of deducible mixins with ring size equal to 4:  0.4846698113207547
Persentage of deducible mixins with ring size equal to 10:  0.62


In [15]:
inputs_mul_mix[['input_value', 'key_offset', 'transaction_hash', 'input_mixin']][inputs_mul_mix['input_mixin'] == '9'][:107]

,input_value,key_offset,transaction_hash,input_mixin
740,10000000000000,"[123, 31, 47, 21, 140, 15, 35, 2, 102, 26]",b9471cec2574358d2333bdc16a795395f3df5057c04e63...,9
741,10000000000000,"[195, 6, 84, 92, 19, 76, 13, 35, 61, 9]",dbc0a739b9782559633fa659fbee309719e38408147a78...,9
742,500000000000,"[196, 15, 73, 37, 16, 80, 19, 10, 17, 3]",dbc0a739b9782559633fa659fbee309719e38408147a78...,9
743,7000000000000,"[178, 2, 26, 24, 37, 25, 14, 135, 54, 3]",dbc0a739b9782559633fa659fbee309719e38408147a78...,9
744,80000000000,"[63, 56, 21, 14, 35, 17, 7, 5, 4, 17]",d5feea5c6959737638cbe6ebb75972e79a5a555e7e34ec...,9
...,...,...,...,...
842,6000000000000,"[7, 13, 3, 3, 3, 14, 4, 5, 2, 9]",895cbf3e5fc584f0c91d2c9ae847f8829191979d6def3f...,9
843,5000000000,"[49, 1, 13, 4, 3, 9, 7, 10, 3, 14]",895cbf3e5fc584f0c91d2c9ae847f8829191979d6def3f...,9
844,90000000000,"[110, 13, 35, 19, 15, 22, 7, 9, 11, 52]",895cbf3e5fc584f0c91d2c9ae847f8829191979d6def3f...,9
845,500000000000,"[157, 65, 36, 50, 30, 46, 27, 48, 40, 26]",895cbf3e5fc584f0c91d2c9ae847f8829191979d6def3f...,9


In [24]:

for _, row in inputs_mul_mix[['input_value', 'key_offset', 'transaction_hash', 'input_mixin']][inputs_mul_mix['input_mixin'] == '9'][107:].iterrows():
    key_offset = row.key_offset.split('[')[1].split(']')[0].split(',')
    value = row.input_value
    num_mixin = row.input_mixin
    for i, key in enumerate(key_offset):
        total_mixins[str(num_mixin)] += 1
        key_i = int(key)
        out = get_transactions(key_i, int(value))
        if out['outs'][0]['key'] in tx_used:
            if num_mixin == 1:
                # If a transaction with 2 inputs uses one of the used inputs of our list,
                # we can also addthe other input in the list since it will be another used output
                tx_used.append(get_transactions(int(key_offset[1-i]), int(value))['outs'][0]['key'])
            vul_mixins[str(num_mixin)] += 1
            4
            print('-----------')
            print('Vul found')

-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
-----------
Vul found
----------

In [25]:
print("Persentage of deducible mixins with ring size equal to 10: ", vul_mixins['9']/total_mixins['9'])

Persentage of deducible mixins with ring size equal to 10:  0.62
